# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

### Cách chọn subset (khác bản gốc — có chủ đích)

Bản gốc chọn `df.sample(1500, random_state=SEED)` ngẫu nhiên và
`chunks_df.head(400)` theo thứ tự file. Với golden dataset trải trên row 33..4997,
cách đó chỉ giữ **19/51** article evidence → phần lớn câu hỏi eval thành
không-thể-trả-lời và phép so sánh Flat vs Graph mất ý nghĩa.

Bản này giữ **nguyên số** của scale guard nhưng chọn mẫu **deterministic + evidence-aware**:

1. `standardize_news()` — giữ trọn article evidence của golden dataset, lấp phần còn lại
   tới đủ 1500 bằng shuffle theo `SEED` (làm distractor cho retrieval).
2. `select_extraction_source()` — trong budget 400 chunk, nạp chunk của article evidence
   trước, còn chỗ mới lấp bằng chunk đầu file.

Hệ quả cần biết khi đọc kết quả: **Flat RAG index phủ toàn bộ ~3000 chunk**, còn
**graph chỉ phủ 400 chunk đã extract**. Tức Flat RAG được truy cập tập văn bản rộng hơn
(superset) — cách đọc này *bất lợi* cho GraphRAG, nên chênh lệch nghiêng về GraphRAG
không phải do graph được ưu ái về dữ liệu.


# PHẦN 1 — SETUP & PREPROCESSING

### Secrets — Colab Secrets HOẶC file `.env` ở gốc repo

Notebook đọc secret theo thứ tự: **Colab `userdata` → `.env` → biến môi trường**.
Chạy local thì `cp .env.example .env` rồi điền:

- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`
- `GROQ_API_KEY`, `GROQ_MODEL` — dùng cho coref, NER+RE, seed extraction, generator
- `JUDGE_PROVIDER`, `JUDGE_MODEL` (+ `OPENAI_API_KEY` nếu judge là OpenAI)
- `HF_TOKEN` để stream dataset từ Hugging Face

`.env` đã nằm trong `.gitignore`. **Không hard-code API key vào notebook nộp bài.**

### Neo4j local bằng Docker (thay cho AuraDB)

```bash
docker run -d --name lab19-neo4j -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/<password-cua-ban> \
  -e NEO4J_server_memory_heap_max__size=2G \
  -e NEO4J_server_memory_pagecache_size=1G \
  neo4j:5.26
```

Rồi đặt `NEO4J_URI=bolt://localhost:7687` trong `.env`.


In [1]:
#@title 1.1 — Install (chỉ cài package còn thiếu)
# Bản gốc cài thêm spacy / langchain-community / llama-index nhưng KHÔNG cell nào dùng
# -> đã loại bỏ (tiết kiệm vài phút mỗi lần Run All, tránh xung đột dependency).
# Thêm python-dotenv để đọc .env khi chạy local (thay cho Colab Secrets).
import importlib.util

REQUIRED = {
    "neo4j": "neo4j",
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "groq": "groq",
    "openai": "openai",
    "tqdm": "tqdm",
    "networkx": "networkx",
    "datasets": "datasets",
    "dotenv": "python-dotenv",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    spec = " ".join(missing)
    print("Đang cài:", spec)
    %pip -q install $spec
else:
    print("✅ Dependencies đã đủ — bỏ qua bước cài.")


✅ Dependencies đã đủ — bỏ qua bước cài.


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

# --- Project layout portable: chạy được cả Colab lẫn local (Windows/Linux) ---
# Bản gốc hard-code /content/... nên crash ngoài Colab và không xuất CSV vào outputs/.
def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "RUBRIC.md").exists() or (cand / ".git").exists():
            return cand
    return here

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"
for _d in (DATA_DIR, OUTPUTS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Secrets: Colab userdata -> .env -> os.environ. Không hard-code key vào notebook. ---
def _read_dotenv(path: Path) -> dict:
    if not path.exists():
        return {}
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip().strip('"').strip("'")
    return out

DOTENV = _read_dotenv(PROJECT_ROOT / ".env")

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or DOTENV.get(name) or default

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

# --- Paths (override được qua .env nếu muốn) ---
DATA_PATH = Path(get_secret("DATA_PATH") or (DATA_DIR / "hackernoon_subset.csv"))
GOLDEN_PATH = Path(get_secret("GOLDEN_PATH") or (DATA_DIR / "graphrag_golden_50_first5000.csv"))
GOLDEN_DETAILED_PATH = DATA_DIR / "graphrag_golden_50_first5000_detailed.csv"
CHECKPOINT = OUTPUTS_DIR / "graphrag_eval_checkpoint.csv"

# --- Scale guard (giữ nguyên theo yêu cầu lab) ---
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

# Golden dataset được soạn trên 5000 dòng ĐẦU của hackernoon_subset.csv,
# nên chỉ stream đúng 5000 dòng đó để row index khớp tuyệt đối.
GOLDEN_SOURCE_ROWS = 5000

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_PATH    = {DATA_PATH}")
print(f"GOLDEN_PATH  = {GOLDEN_PATH}")
print("Secrets:", {
    k: ("✅" if get_secret(k) else "❌")
    for k in ["NEO4J_URI", "NEO4J_PASSWORD", "GROQ_API_KEY", "GROQ_MODEL",
              "JUDGE_PROVIDER", "JUDGE_MODEL", "HF_TOKEN"]
})


PROJECT_ROOT = E:\WED\K3-Track3-Lab19-GraphRAG
DATA_PATH    = E:\WED\K3-Track3-Lab19-GraphRAG\data\hackernoon_subset.csv
GOLDEN_PATH  = E:\WED\K3-Track3-Lab19-GraphRAG\data\graphrag_golden_50_first5000.csv
Secrets: {'NEO4J_URI': '✅', 'NEO4J_PASSWORD': '✅', 'GROQ_API_KEY': '✅', 'GROQ_MODEL': '✅', 'JUDGE_PROVIDER': '✅', 'JUDGE_MODEL': '✅', 'HF_TOKEN': '✅'}


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới stream dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV,
nên không cần nạp toàn bộ dataset vào RAM.

### Giới hạn đang dùng

- `LIMIT_ROWS = GOLDEN_SOURCE_ROWS = 5000` — **chốt cứng 5000 dòng đầu**.
- `PRIORITIZE_MB = False` — dừng theo **số dòng**, không theo dung lượng.
- `LIMIT_MB = 300` — chỉ còn là chặn an toàn.

> **Vì sao đúng 5000 dòng?** Golden dataset (`data/graphrag_golden_50_first5000*.csv`)
> được soạn trên đúng 5000 dòng đầu của file này, cột `evidence_row_ids_0based` trỏ tới
> row index 0-based trong đó. Cell này ghi **tuần tự theo thứ tự stream**, nên tải lại
> 5000 dòng đầu sẽ tái lập chính xác cùng tập article — evidence vẫn khớp.

### Lưu ý

- `HF_TOKEN` đọc từ `.env` (local) hoặc Colab Secrets — không hard-code.
- Dataset là **gated**: phải mở trang dataset trên Hugging Face và hoàn tất
  **Agree / Request access**; token dạng `fineGrained` cần bật scope
  *"Read access to contents of all public gated repos you can access"*.
- Cell **idempotent**: nếu `DATA_PATH` đã có đủ 5000 dòng thì bỏ qua download.
  Muốn tải lại, đặt `FORCE_REDOWNLOAD = True`.
- Đích ghi là `DATA_PATH` (mặc định `data/hackernoon_subset.csv` trong repo),
  không còn `/content/...` như bản gốc.


In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = Path(DATA_PATH)

# Golden dataset tham chiếu row 0..4999 của file này => chốt LIMIT_ROWS = 5000.
# Cell 006 ghi tuần tự theo thứ tự stream nên row index tái lập được chính xác.
LIMIT_ROWS = GOLDEN_SOURCE_ROWS
LIMIT_MB = 300              # chặn an toàn, không phải mục tiêu
PRIORITIZE_MB = False       # dừng theo số dòng để khớp golden dataset
FORCE_REDOWNLOAD = False    # True nếu muốn tải lại từ đầu


def _csv_rows(path: Path) -> int:
    with open(path, encoding="utf-8", newline="") as f:
        return max(0, sum(1 for _ in csv.reader(f)) - 1)


# Idempotent: Run All nhiều lần không tải lại 5000 dòng.
if OUTPUT_CSV.exists() and not FORCE_REDOWNLOAD and _csv_rows(OUTPUT_CSV) >= LIMIT_ROWS:
    print(f"✅ Đã có {OUTPUT_CSV} ({_csv_rows(OUTPUT_CSV):,} dòng) — bỏ qua download.")
else:
    if not HF_TOKEN:
        raise ValueError(
            "Thiếu HF_TOKEN. Đặt trong .env (local) hoặc Colab Secrets. "
            "Dataset là gated: phải bấm Agree/Request access trên trang Hugging Face "
            "và token cần scope đọc public gated repos."
        )

    print("Đang kết nối luồng dữ liệu (streaming)...")
    dataset = load_dataset(DATASET_NAME, split="train", streaming=True, token=HF_TOKEN)
    iterator = iter(dataset)

    try:
        first_row = next(iterator)
    except StopIteration:
        raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")

    headers = list(first_row.keys())
    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(total=total_progress, desc=f"Đang tải ({unit_progress})", unit=unit_progress) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                if PRIORITIZE_MB and (rows_written % 100 == 0 or file_size_mb >= LIMIT_MB * 0.95):
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(f"\n[DỪNG] Đạt giới hạn dung lượng {file_size_mb:.2f} MB ({rows_written:,} dòng)")
                    break

                if rows_written >= LIMIT_ROWS:
                    print(f"\n[DỪNG] Đạt giới hạn số dòng: {rows_written:,}")
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(f"✅ Hoàn thành: {OUTPUT_CSV}\n   Rows: {rows_written:,}\n   Size: {final_size_mb:.2f} MB")


✅ Đã có E:\WED\K3-Track3-Lab19-GraphRAG\data\hackernoon_subset.csv (5,000 dòng) — bỏ qua download.


In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.


✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None


def load_golden_evidence_rows():
    """Row index (0-based) của các article được golden dataset dùng làm evidence.

    Vì sao cần: golden dataset trải trên row 33..4997 nhưng LAB_MAX_ARTICLES=1500.
    Nếu sample ngẫu nhiên như bản gốc thì chỉ 19/51 article evidence trụ lại
    => phần lớn câu hỏi thành không-thể-trả-lời và phép so sánh mất ý nghĩa.
    """
    if not GOLDEN_DETAILED_PATH.exists():
        print(f"⚠️  Không thấy {GOLDEN_DETAILED_PATH.name} — sẽ sample ngẫu nhiên như bản gốc.")
        return set()
    d = pd.read_csv(GOLDEN_DETAILED_PATH)
    rows = set()
    for s in d.get("evidence_row_ids_0based", pd.Series(dtype=str)).dropna():
        try:
            rows.update(int(x) for x in json.loads(s))
        except Exception:
            continue
    print(f"Golden evidence: {len(rows)} article (row {min(rows)}..{max(rows)})" if rows else "Golden evidence: 0")
    return rows


GOLDEN_EVIDENCE_ROWS = load_golden_evidence_rows()


def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in {".jsonl", ".ndjson"}:
        df = pd.read_json(path, lines=True)
    elif path.suffix.lower() == ".json":
        df = pd.read_json(path)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported: {path.suffix}")
    # Giữ vị trí gốc để đối chiếu evidence_row_ids_0based của golden dataset.
    df["_source_row"] = np.arange(len(df))
    return df


def standardize_news(raw):
    # Dataset HackerNoon thật dùng cột `description` làm body bài báo
    # (schema: companyName, companyUrl, published_at, url, title, main_image, description).
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df["source_row"] = (
        raw["_source_row"].to_numpy() if "_source_row" in raw.columns else np.arange(len(raw))
    )

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        # Sampling deterministic + evidence-aware: giữ TRỌN article evidence của golden
        # dataset, phần còn lại lấp bằng shuffle theo SEED để làm distractor.
        # Vẫn tôn trọng scale guard LAB_MAX_ARTICLES.
        must = df["source_row"].isin(GOLDEN_EVIDENCE_ROWS)
        keep = df[must]
        room = max(0, LAB_MAX_ARTICLES - len(keep))
        fill = df[~must].sample(frac=1.0, random_state=SEED).head(room)
        df = (
            pd.concat([keep, fill])
            .sort_values("source_row")
            .reset_index(drop=True)
        )
        print(
            f"Sample {LAB_MAX_ARTICLES}: {len(keep)} evidence article + {len(fill)} distractor"
        )
        missing = GOLDEN_EVIDENCE_ROWS - set(df["source_row"])
        if missing:
            print(f"⚠️  {len(missing)} evidence row bị loại ở bước dedup/độ dài: {sorted(missing)[:20]}")
    return df


def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out


def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "source_row": int(r.source_row),
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                print(f"⚠️  Đạt LAB_MAX_CHUNKS={LAB_MAX_CHUNKS}, dừng chunking sớm.")
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)


raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
print(f"news_df={len(news_df):,} article | chunks_df={len(chunks_df):,} chunk")
display(chunks_df.head())


Golden evidence: 51 article (row 33..4997)


Exact dedup: 2,675 -> 2,105
Sample 1500: 51 evidence article + 1449 distractor


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

news_df=1,500 article | chunks_df=1,500 chunk


,chunk_id,article_id,source_row,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,0,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,2,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
2,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,3,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
3,6633c15d86f5e81f47b8::c0000,6633c15d86f5e81f47b8,27,5 Kubernetes technology vendors hot right now,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and it is now a key plank in the Dev...
4,4ce72a4490a6a618e2d5::c0000,4ce72a4490a6a618e2d5,28,Bachelor of Science in Health Information Management,2023-08-16,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

def select_extraction_source(chunks_df, news_df, limit=None):
    """Chọn <=limit chunk để coref + NER/RE, ưu tiên article evidence của golden dataset.

    Bản gốc lấy chunks_df.head(400) theo thứ tự file. Với budget 400 chunk, cách đó
    khiến graph gần như không phủ được câu hỏi eval (evidence trải tới row 4997).
    Ở đây: nạp trọn chunk của article evidence trước, còn chỗ thì lấp bằng chunk
    đầu file làm nhiễu. Thứ tự trong mỗi nhóm giữ nguyên -> deterministic.
    """
    limit = int(limit or EXTRACTION_MAX_CHUNKS)
    ev_articles = set(news_df.loc[news_df.source_row.isin(GOLDEN_EVIDENCE_ROWS), "article_id"])
    is_ev = chunks_df.article_id.isin(ev_articles)

    prio = chunks_df[is_ev]
    if len(prio) > limit:
        print(f"⚠️  Chunk evidence ({len(prio)}) > EXTRACTION_MAX_CHUNKS ({limit}) — sẽ bị cắt.")
    fill = chunks_df[~is_ev].head(max(0, limit - len(prio)))
    sel = pd.concat([prio, fill]).head(limit).reset_index(drop=True)

    covered = sel.article_id.nunique()
    ev_covered = sel.loc[sel.article_id.isin(ev_articles), "article_id"].nunique()
    print(
        f"extraction_source: {len(sel)} chunk / {covered} article "
        f"| evidence article phủ được {ev_covered}/{len(ev_articles)}"
    )
    return sel


extraction_source = select_extraction_source(chunks_df, news_df).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
print(
    "Chunk coref thất bại:",
    int(extraction_source.unresolved_mentions.apply(
        lambda v: isinstance(v, list) and "COREF_BATCH_FAILED" in v
    ).sum()),
)
display(extraction_source[["chunk_id", "text", "resolved_text"]].head(3))

extraction_source: 400 chunk / 400 article | evidence article phủ được 51/51


Coref:   0%|          | 0/80 [00:00<?, ?it/s]

Chunk coref thất bại: 0


,chunk_id,text,resolved_text
0,4f1346392056a403277d::c0000,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry Ericsson'...,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry Ericsson'...
1,8c5930949a3d9f3c3a38::c0000,Samsung Electronics Co. Ltd. a world leader in advanced semiconductor technology today unveiled its latest innovatio...,Samsung Electronics Co. Ltd. a world leader in advanced semiconductor technology today unveiled its latest innovatio...
2,ebe57b3790f81ceb3734::c0000,HE Saeed Mohammed Al Tayer MD & CEO of Dubai Electricity and Water Authority (DEWA) announced that DEWA is working t...,HE Saeed Mohammed Al Tayer MD & CEO of Dubai Electricity and Water Authority (DEWA) announced that DEWA is working t...


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())

NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Aeris Communications,Company,PARTNERED_WITH,Ericsson,Company,4f1346392056a403277d::c0000,2022-12-07,Aeris Communications and Ericsson are joining together,1.0
1,Samsung Electronics Co. Ltd.,Company,DEVELOPED,analog semiconductor technologies,Technology,8c5930949a3d9f3c3a38::c0000,2023-10-05,unveiled its latest innovations in analog ... semiconductor technologies,1.0
2,Samsung Electronics Co. Ltd.,Company,DEVELOPED,logic semiconductor technologies,Technology,8c5930949a3d9f3c3a38::c0000,2023-10-05,unveiled its latest innovations in ... logic semiconductor technologies,1.0
3,Saeed Mohammed Al Tayer,Person,LEADS,Dubai Electricity and Water Authority,Company,ebe57b3790f81ceb3734::c0000,2023-02-09,HE Saeed Mohammed Al Tayer MD & CEO of Dubai Electricity and Water Authority (DEWA),1.0
4,Dubai Electricity and Water Authority,Company,USES,ChatGPT,Technology,ebe57b3790f81ceb3734::c0000,2023-02-09,DEWA is working to enrich DEWA's services with ChatGPT technology,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,type,left,right,similarity,decision
0,Technology,generative AI,Generative AI features,0.906581,MERGE_VECTOR


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 113, 'edges': 79, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,8
1,cc9c6ee3857729e221d3f6de,ServiceNow,Company,5
2,909fcd9c188c8c2429afa468,Google Cloud,Company,4
3,35c663762dedf469a841fdb9,Dell Technologies Inc.,Company,3
4,3dd22ce4097a0a751551ba96,Generative AI features,Technology,3
5,1a324b330bd048e7308e53f0,Artificial Intelligence,Technology,3
6,ce0b6a8e50a6a58526e4c85f,Aeris,Company,3
7,8d3a4b002f6b4a95d8e04dc9,AI Lighthouse,Technology,3
8,b823fb02e58a12bdc4c38934,NVIDIA,Company,2
9,02f50203f5a967ac3ebb42be,Ericsson,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [14]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [15]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [16]:
#@title 4.1 — 5 câu Golden starter
starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

# GOLDEN_PATH được định nghĩa ở cell 1.2 (config), trỏ vào data/ trong repo.
# File thật: 50 câu hard, soạn trên 5000 row đầu của hackernoon_subset.csv,
# đủ 3 nhóm factoid / multi-hop / cross-doc và đã điền reference_answer.
if Path(GOLDEN_PATH).exists():
    golden_df = pd.read_csv(GOLDEN_PATH)
    print(f"✅ Golden dataset: {GOLDEN_PATH.name} — {len(golden_df)} câu")
else:
    golden_df = starter_golden.copy()
    print(f"⚠️  Không thấy {GOLDEN_PATH} — dùng 5 câu starter (G02..G05 chưa có gold answer).")

print(golden_df.group.value_counts().to_dict())
display(golden_df.head())

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

✅ Golden dataset: graphrag_golden_50_first5000.csv — 50 câu
{'multi-hop': 23, 'cross-doc': 22, 'factoid': 5}


,id,group,question,reference_answer,reference_evidence
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...


In [17]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [18]:
#@title 4.3 — Evaluation runner + checkpoint
# CHECKPOINT được định nghĩa ở cell 1.2 -> outputs/graphrag_eval_checkpoint.csv
def run_evaluation(golden_df, resume=True):
    """Chạy eval cho từng câu. resume=True -> bỏ qua id đã có trong checkpoint.

    Với 50 câu (~200 lệnh gọi LLM), resume là bắt buộc để không phải chạy lại từ đầu
    khi gặp rate limit / mất mạng.
    """
    rows = []
    done = set()
    if resume and Path(CHECKPOINT).exists():
        prev = pd.read_csv(CHECKPOINT)
        rows = prev.to_dict("records")
        done = set(prev["id"].astype(str))
        print(f"Resume từ checkpoint: đã có {len(done)} câu.")
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        if str(q.id) in done:
            continue
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.


Evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

RuntimeError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kyw7pmjnfzasmmffmswzsefj` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199740, Requested 1386. Please try again in 8m6.431999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)

# RUBRIC 3.3 ghi reports/, README + ASSIGNMENT ghi outputs/ -> xuất cả 2 nơi cho chắc.
for _dir in (OUTPUTS_DIR, REPORTS_DIR):
    eval_results_df.to_csv(_dir / "graphrag_eval_results.csv", index=False)
    comparison_df.to_csv(_dir / "graphrag_vs_flatrag_summary.csv", index=False)
    print(f"✅ Đã ghi {_dir.name}/graphrag_eval_results.csv + graphrag_vs_flatrag_summary.csv")

# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [ ]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau